# 15 · Crop geometry, noise ceilings and equivalence: BV vs VEDB

**Question.** BV and VEDB cluster CDI domains equally (AUC difference ≈ 0 in CLIP and DINOv3). Is that because
infant and adult head-cam crops look the same, or do they differ in how objects are framed, with structure that
survives anyway? And how much of the remaining BV–VEDB difference is real rather than sampling noise?

**Five analyses**
1. **Describe crop geometry** per category and domain: size, resolution, vertical position, truncation at the
   frame edge, aspect ratio, boxes per frame, detector confidence. Differences get a unit bootstrap
   (BV subjects, VEDB sessions).
2. **AUC within geometry levels**: size tertiles, top / middle / bottom third of the frame, truncated or not.
   Does clustering depend on how close or how cut-off the views are?
3. **Geometry matching**: reweight each category's crops so BV and VEDB have the same distribution over
   size × vertical position × truncation cells, then recompute AUC and the BV–VEDB RDM correlation.
4. **Noise ceilings**: split-half RDM reliability (BV by subject, VEDB by session), Spearman–Brown corrected.
   The BV–VEDB correlation is read against √(rel_BV · rel_VEDB), before and after matching.
5. **Equivalence**: BV − VEDB AUC with a crossed unit × category bootstrap, tested against ±`EQ_BOUND`
   (two one-sided tests = 90% interval inside the bound).

**Decisions**
- Same crops as the manuscript and notebook 11: 2025.2 packed BV shards and VEDB shards, CLIP image–label
  similarity ≥ 0.27, valid129 categories, crops L2-normalized; a check cell confirms the category sums match
  notebook 11's cache.
- Frames are fixed per dataset (checked): BV 512 × 910 portrait, VEDB 2048 × 1536 landscape. All positions and sizes
  are fractions of the frame. Resolution (box short side in pixels) is reported but not matched: VEDB crops are
  sharper at every size, and the encoder resizes crops anyway.
- Truncated = box within `EDGE_FRAC` of any frame edge. Size bins = sextiles of log₁₀(box area / frame area),
  pooled with equal weight per dataset, so tertiles are pairs of sextiles.
- Matching uses common cell weights per category, the average of the two datasets' cell shares over cells where
  both have ≥ `MIN_CELL` crops. Means are z-scored across categories within each condition (manuscript convention).
- Category set: ≥ `MIN_N` crops in both datasets for both models and ≥ 5 per half in every split.
- Bootstrap draws (within-domain category resampling, self-pairs dropped) are shared across datasets and conditions.

**Outputs:** `outputs/crop_geometry/results/` and `figures/`. The geometry tables and binned sums go to
`outputs/crop_geometry/cache/` (git-ignored; rebuilt in ~30 min when missing or `RECOMPUTE_CACHE = True`).

In [ ]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

# Manuscript figure style (same as analysis/qwen-open-vocab/fig_style.py): Lato, bold axis labels and legends.
import matplotlib.axes, matplotlib.axis, matplotlib.figure
from matplotlib import font_manager
for _f in (Path.cwd().resolve().parents[1] / "talk-figures" / "unr-2026" / "fonts").glob("*.ttf"):
    font_manager.fontManager.addfont(str(_f))
plt.rcParams.update({
    "font.family": ["Lato", "DejaVu Sans", "sans-serif"], "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.labelweight": "bold", "figure.titlesize": 12, "figure.titleweight": "bold",
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#444444", "axes.labelcolor": "#222222",
    "text.color": "#222222", "xtick.color": "#444444", "ytick.color": "#444444"})
def _bold_legend(orig):
    if getattr(orig, "_bold", False):
        return orig
    def legend(self, *args, **kw):
        prop = dict(kw.pop("prop", {}) or {}); prop.setdefault("weight", "bold")
        prop.setdefault("size", kw.pop("fontsize", plt.rcParams["legend.fontsize"]))
        return orig(self, *args, prop=prop, **kw)
    legend._bold = True
    return legend
matplotlib.axes.Axes.legend = _bold_legend(matplotlib.axes.Axes.legend)
matplotlib.figure.Figure.legend = _bold_legend(matplotlib.figure.Figure.legend)
_tick_init = matplotlib.axis.Tick.__init__
if not getattr(_tick_init, "_bold", False):
    def _bold_tick(self, *a, **k):                       # bold tick labels (no rcParam for tick label weight)
        _tick_init(self, *a, **k); self.label1.set_fontweight("bold"); self.label2.set_fontweight("bold")
    _bold_tick._bold = True; matplotlib.axis.Tick.__init__ = _bold_tick
    _grid = matplotlib.axes.Axes.grid                    # no grid lines in any figure
    matplotlib.axes.Axes.grid = lambda self, visible=None, which="major", axis="both", **kw: _grid(self, False, which=which, axis=axis)
plt.rcParams["axes.grid"] = False
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [ ]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

# Inputs for this notebook (read only).
MANUSCRIPT_SCRIPTS = MANUSCRIPT_DIR / "scripts"
PACKED = {"bv": Path("/ccn2a/dataset/babyview/2025.2/outputs/image_embeddings/babyview_full"),
          "vedb": Path("/ccn2a/dataset/babyview/related_data/vedb_1612/outputs/image_embeddings/vedb_full_adults")}
CTX_CACHE = OUT / "context_split" / "cache"        # notebook 11 per-video sums, used only for the check

GEO_OUT = OUT / "crop_geometry"
GEO_CACHE, GEO_RESULTS, GEO_FIGURES = GEO_OUT / "cache", GEO_OUT / "results", GEO_OUT / "figures"
for d in (GEO_CACHE, GEO_RESULTS, GEO_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

## Parameters

In [ ]:
MIN_SIM = 0.27           # CLIP image–label threshold (manuscript)
EXCLUDE_BV_SUBJECTS = ("00270001",)
FRAME_WH = {"bv": (512, 910), "vedb": (2048, 1536)}
EDGE_FRAC = 0.01         # truncated: box within 1% of the frame width/height from an edge
N_SIZE, N_VPOS = 6, 3    # size sextiles × top/middle/bottom thirds × truncated (2) = 36 cells
MIN_N = 20               # crops per category per dataset (both models)
MIN_HALF = 5             # crops per category in each half of every split
MIN_LEVEL = 10           # crops per category in every level of a geometry measure (analysis 2)
MIN_CELL = 3             # crops per dataset for a cell to count in matching
MIN_MATCHED = 20         # matched crops per category per dataset
K_SPLITS = 20            # random unit half splits (same seeds as notebook 11)
N_BOOT = 1000            # bootstrap draws
EQ_BOUND = 0.04          # equivalence bound on the BV − VEDB AUC difference
RECOMPUTE_CACHE = False

DATASETS = ("bv", "vedb")
LABEL = {"bv": "BabyView", "vedb": "VEDB"}
COLOR = {"bv": "#0D47A1", "vedb": "#E07B39"}   # same as notebooks 07-11
MLABEL = {"clip": "CLIP", "dinov3": "DINOv3"}

## Shared helpers

In [ ]:
import sys
sys.path.insert(0, str(MANUSCRIPT_SCRIPTS))
from packed_crop_embeddings import (_row_mask, _vectors_for_mask, iter_shard_paths, l2_normalize_rows,
                                    subject_id_from_video_id)


def load_cdi_semantic_map() -> dict[str, str]:
    """``uni_lemma`` → CDI semantic domain from ``data/cdi_words.csv`` (same as notebooks 09 and 11)."""
    df = pd.read_csv(DATA_DIR / "cdi_words.csv", usecols=["uni_lemma", "category"])
    df["uni_lemma"] = df["uni_lemma"].astype(str).str.strip().str.lower()
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.drop_duplicates(subset=["uni_lemma"], keep="first").set_index("uni_lemma")["category"].to_dict()


CATS = sorted(l.strip().lower() for l in VALID129_TXT.read_text().splitlines() if l.strip())
CIX = {c: i for i, c in enumerate(CATS)}
CDI = load_cdi_semantic_map()
K_CELLS = N_SIZE * N_VPOS * 2


def unit_of(ds: str, vid: str) -> str:
    """Resampling unit. BV: subject ID (one child per ID). VEDB: session."""
    return vid.split("_")[0] if ds == "bv" else vid


def split_halves(units: np.ndarray, k: int) -> np.ndarray:
    """Boolean over sorted ``units``: True = half A of split k (same seeds as notebook 11)."""
    rng = np.random.default_rng(1000 + k); in_a = set(rng.permutation(units)[: len(units) // 2])
    return np.array([u in in_a for u in units])


def zscore_rows(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def rdm(M: np.ndarray) -> np.ndarray:
    Z = zscore_rows(M); Z = Z / np.linalg.norm(Z, axis=1, keepdims=True); return 1 - Z @ Z.T


def pair_values(R, sem, idx):
    """Upper-triangle distances over (possibly resampled) categories, self-pairs dropped."""
    s = np.asarray(sem)[idx]; r = R[np.ix_(idx, idx)]; iu = np.triu_indices(len(idx), 1)
    keep = idx[iu[0]] != idx[iu[1]]
    return r[iu][keep], (s[iu[0]] == s[iu[1]])[keep]


def auc(R, sem, idx=None):
    """P(between-domain distance > within-domain distance) over category pairs; ties count half."""
    idx = np.arange(len(sem)) if idx is None else idx
    v, same = pair_values(R, sem, idx); w, b = v[same], v[~same]
    ranks = rankdata(np.concatenate([w, b]))
    return (ranks[len(w):].sum() - len(b) * (len(b) + 1) / 2) / (len(w) * len(b))


def rdm_rho(R1, R2, idx=None):
    idx = np.arange(len(R1)) if idx is None else idx
    iu = np.triu_indices(len(idx), 1); keep = idx[iu[0]] != idx[iu[1]]
    a = R1[np.ix_(idx, idx)][iu][keep]; b = R2[np.ix_(idx, idx)][iu][keep]
    return spearmanr(a, b)[0]


def domain_bootstrap(sem, n, seed):
    rng = np.random.default_rng(seed); sem = np.asarray(sem); doms = sorted(set(sem))
    return [np.concatenate([rng.choice(np.flatnonzero(sem == dd), (sem == dd).sum()) for dd in doms])
            for _ in range(n)]


def group_sum(keys: np.ndarray, X: np.ndarray, n_groups: int) -> np.ndarray:
    """Row sums of ``X`` per integer key (sparse indicator product; much faster than np.add.at)."""
    from scipy.sparse import csr_matrix
    G = csr_matrix((np.ones(len(keys), np.float32), (keys, np.arange(len(keys)))), shape=(n_groups, len(keys)))
    return np.asarray(G @ X)


def ci_p(x, level=95):
    x = np.asarray(x)[np.isfinite(x)]; a = (100 - level) / 2
    lo, hi = np.percentile(x, [a, 100 - a]); return lo, hi, float(2 * min((x <= 0).mean(), (x >= 0).mean()))

## Stage A · crop geometry from the crop indexes

One row per analysis crop (valid129, CLIP ≥ 0.27). Boxes per frame counts every detection in that frame, any class
or score, so it measures clutter as the detector saw it.

In [ ]:
INDEX_COLS = ["image_id", "video_id", "frame_number", "class_name", "xmin", "ymin", "xmax", "ymax",
              "yolo_conf", "clip_sim", "has_dino"]


def load_geometry(ds: str) -> pd.DataFrame:
    W, H = FRAME_WH[ds]; parts, nbox = [], []
    for ch in pd.read_csv(PACKED[ds] / "crops_index.csv.gz", usecols=INDEX_COLS, chunksize=2_000_000):
        assert ch["xmax"].max() <= W + 1 and ch["ymax"].max() <= H + 1, f"{ds}: box outside {W}x{H} frame"
        nbox.append(ch.groupby(["video_id", "frame_number"]).size())
        ch["class_name"] = ch["class_name"].astype(str).str.strip().str.lower()
        parts.append(ch[ch["class_name"].isin(set(CATS)) & (ch["clip_sim"] >= MIN_SIM)])
    g = pd.concat(parts, ignore_index=True)
    if ds == "bv":
        sid = {v: subject_id_from_video_id(v) for v in g["video_id"].unique()}
        g = g[~g["video_id"].map(sid).isin(EXCLUDE_BV_SUBJECTS)].reset_index(drop=True)
    nb = pd.concat(nbox).groupby(level=[0, 1]).sum()
    g["n_frame_boxes"] = nb.reindex(pd.MultiIndex.from_frame(g[["video_id", "frame_number"]])).to_numpy()
    w = (g["xmax"] - g["xmin"]).clip(lower=1.0); h = (g["ymax"] - g["ymin"]).clip(lower=1.0)
    ex, ey = EDGE_FRAC * W, EDGE_FRAC * H
    return pd.DataFrame({
        "image_id": g["image_id"].astype(str), "video_id": g["video_id"].astype(str),
        "unit": [unit_of(ds, v) for v in g["video_id"].astype(str)],
        "cat": g["class_name"].map(CIX).astype(np.int16), "has_dino": g["has_dino"].astype(bool),
        "log_area": np.log10(w * h / (W * H)), "short_px": np.minimum(w, h), "log_aspect": np.log2(w / h),
        "cx": (g["xmin"] + g["xmax"]) / 2 / W, "cy": (g["ymin"] + g["ymax"]) / 2 / H,
        "trunc": ((g["xmin"] <= ex) | (g["ymin"] <= ey) | (g["xmax"] >= W - ex) | (g["ymax"] >= H - ey)),
        "n_frame_boxes": g["n_frame_boxes"].astype(np.int32), "yolo_conf": g["yolo_conf"].astype(np.float32)})


GEO_FILE = {ds: GEO_CACHE / f"{ds}_crop_geometry.pkl" for ds in DATASETS}
for ds in DATASETS:
    if RECOMPUTE_CACHE or not GEO_FILE[ds].exists():
        load_geometry(ds).to_pickle(GEO_FILE[ds])
GEO = {ds: pd.read_pickle(GEO_FILE[ds]) for ds in DATASETS}
for ds in DATASETS:
    print(f"{LABEL[ds]:5s} {len(GEO[ds]):>10,} crops, {GEO[ds].unit.nunique()} units, "
          f"{GEO[ds].video_id.nunique():,} videos, truncated {GEO[ds].trunc.mean():.1%}")

### Geometry cells

In [ ]:
# Size sextile edges pooled with equal weight per dataset (each dataset's own quantiles averaged).
SIZE_EDGES = np.mean([np.quantile(GEO[ds]["log_area"], np.linspace(0, 1, N_SIZE + 1)[1:-1]) for ds in DATASETS], axis=0)
VPOS_EDGES = np.array([1 / 3, 2 / 3])
for ds in DATASETS:
    g = GEO[ds]
    g["size_bin"] = np.searchsorted(SIZE_EDGES, g["log_area"], side="right").astype(np.int8)
    g["vpos_bin"] = np.searchsorted(VPOS_EDGES, g["cy"], side="right").astype(np.int8)
    g["cell"] = ((g["size_bin"] * N_VPOS + g["vpos_bin"]) * 2 + g["trunc"].astype(np.int8)).astype(np.int16)
bins_table = pd.DataFrame({
    "size_bin": range(N_SIZE),
    "log10_area_lo": np.r_[-np.inf, SIZE_EDGES], "log10_area_hi": np.r_[SIZE_EDGES, np.inf],
    **{f"share_{ds}": GEO[ds]["size_bin"].value_counts(normalize=True).reindex(range(N_SIZE), fill_value=0).to_numpy()
       for ds in DATASETS}})
bins_table.to_csv(GEO_RESULTS / "geometry_size_bins.csv", index=False)
display(bins_table.round(3))

## Stage B · geometry-binned embedding sums (one pass over the shards)

For every (unit, category, cell) and model, the sum and count of L2-normalized crop vectors. Shard rows are joined to
the geometry table by `image_id`; every analysis crop must find its cell.

In [ ]:
def build_binned_sums(ds: str, out: Path) -> None:
    g = GEO[ds]; cell_of = {v: s for v, s in g.groupby("video_id")[["image_id", "cell"]]}
    units = np.array(sorted(g["unit"].unique())); uix = {u: i for i, u in enumerate(units)}
    rec = {m: {"u": [], "c": [], "k": [], "n": [], "sum": []} for m in MODELS}
    paths = list(iter_shard_paths(PACKED[ds])); n_rows = 0
    for j, p in enumerate(paths):
        sid = subject_id_from_video_id(p.stem)
        if ds == "bv" and sid in EXCLUDE_BV_SUBJECTS:
            continue
        with np.load(p, allow_pickle=True) as z:
            sh = {kk: z[kk] for kk in z.files}
        mc = _row_mask(sh, categories=set(CATS), min_sim=MIN_SIM, passed_only=False, model="clip")
        if not mc.any():
            continue
        look = cell_of[p.stem].set_index("image_id")["cell"]
        cell = look.reindex(sh["image_id"][mc].astype(str)).to_numpy()
        assert not np.isnan(cell.astype(float)).any(), f"{p.stem}: crops missing from the index"
        cell = cell.astype(np.int64); n_rows += int(mc.sum())
        cat = np.array([CIX[str(x).strip().lower()] for x in sh["class_name"][mc]])
        u = uix[unit_of(ds, p.stem)]
        md_ = _row_mask(sh, categories=set(CATS), min_sim=MIN_SIM, passed_only=False, model="dinov3")
        for m in MODELS:
            sub = np.ones(mc.sum(), bool) if m == "clip" else md_[mc]
            X = l2_normalize_rows(_vectors_for_mask(sh, mc if m == "clip" else md_, m))
            key = cat[sub] * K_CELLS + cell[sub]
            uk, inv = np.unique(key, return_inverse=True)
            S = np.zeros((len(uk), X.shape[1])); np.add.at(S, inv, X)
            r = rec[m]; r["u"].append(np.full(len(uk), u)); r["c"].append(uk // K_CELLS); r["k"].append(uk % K_CELLS)
            r["n"].append(np.bincount(inv)); r["sum"].append(S.astype(np.float32))
        if j % 500 == 0:
            print(f"  {ds}: {j}/{len(paths)} shards", flush=True)
    assert n_rows == len(g), f"{ds}: {n_rows} shard crops vs {len(g)} index crops"
    np.savez(out, units=units, **{f"{m}_{f}": np.concatenate(rec[m][f]) for m in rec for f in ("u", "c", "k", "n", "sum")})


BIN_FILE = {ds: GEO_CACHE / f"{ds}_binned_sums.npz" for ds in DATASETS}
for ds in DATASETS:
    if RECOMPUTE_CACHE or not BIN_FILE[ds].exists():
        build_binned_sums(ds, BIN_FILE[ds])
B = {ds: dict(np.load(BIN_FILE[ds])) for ds in DATASETS}
print({ds: BIN_FILE[ds].stat().st_size // 2**20 for ds in DATASETS}, "MB")

### Dense views and the check against notebook 11

In [ ]:
def dense(ds: str, m: str):
    """Unit × category sums/counts and category × cell sums/counts."""
    b = B[ds]; U = len(b["units"]); D = b[f"{m}_sum"].shape[1]
    u, c, k, n, S = (b[f"{m}_{f}"] for f in ("u", "c", "k", "n", "sum"))
    C = len(CATS)
    UC = group_sum(u * C + c, S, U * C).reshape(U, C, D)
    UCn = np.bincount(u * C + c, weights=n, minlength=U * C).reshape(U, C)
    CK = group_sum(c * K_CELLS + k, S, C * K_CELLS).reshape(C, K_CELLS, D).astype(np.float64)
    CKn = np.bincount(c * K_CELLS + k, weights=n, minlength=C * K_CELLS).reshape(C, K_CELLS)
    return dict(UC=UC, UCn=UCn, CK=CK, CKn=CKn, units=b["units"], rows=(u, c, k, n, S))


DN = {(ds, m): dense(ds, m) for ds in DATASETS for m in MODELS}
for ds, f in (("bv", "bv_2025_2_video_category_sums.npz"), ("vedb", "vedb_video_category_sums.npz")):
    if not (CTX_CACHE / f).exists():
        print("notebook 11 cache missing; check skipped:", f); continue
    z = np.load(CTX_CACHE / f)
    for m in MODELS:
        ref = np.zeros((len(CATS), z[f"{m}_sum"].shape[1])); rn = np.zeros(len(CATS))
        np.add.at(ref, z[f"{m}_cat"], z[f"{m}_sum"]); np.add.at(rn, z[f"{m}_cat"], z[f"{m}_n"])
        d = DN[(ds, m)]; mine = d["UC"].sum(0); mn = d["UCn"].sum(0)
        assert np.array_equal(mn, rn), f"{ds} {m}: crop counts differ from notebook 11"
        ok = rn > 0; err = np.abs(mine[ok] / mn[ok, None] - ref[ok] / rn[ok, None]).max()
        print(f"{ds:5s} {m:6s} counts identical; max |Δ mean| = {err:.1e}"); assert err < 1e-5

### Category set

In [ ]:
def half_counts(ds, m, k):
    d = DN[(ds, m)]; a = split_halves(d["units"], k); return d["UCn"][a].sum(0), d["UCn"][~a].sum(0)


ok = np.ones(len(CATS), bool)
for ds in DATASETS:
    for m in MODELS:
        ok &= DN[(ds, m)]["UCn"].sum(0) >= MIN_N
        for k in range(K_SPLITS):
            na, nb = half_counts(ds, m, k); ok &= (na >= MIN_HALF) & (nb >= MIN_HALF)
SET = np.flatnonzero(ok); SEM = [CDI.get(CATS[i], "other") for i in SET]
print(len(SET), "categories;", pd.Series(SEM).value_counts().to_dict())
BOOT = domain_bootstrap(SEM, N_BOOT, 15)

## 1 · Describe crop geometry

Per-category means of each measure over that category's crops; domain values are means over its categories. BV − VEDB
differences get a unit bootstrap (subjects and sessions resampled).

In [ ]:
MEASURES = {"log_area": "log₁₀ box area / frame", "short_px": "box short side (px)", "cy": "box centre height (0 = top)",
            "trunc": "truncated at frame edge", "log_aspect": "log₂ width / height", "n_frame_boxes": "boxes per frame",
            "yolo_conf": "detector confidence"}


def unit_cat_sums(ds):
    g = GEO[ds][GEO[ds]["cat"].isin(SET)]; units = np.array(sorted(GEO[ds]["unit"].unique()))
    ui = pd.Index(units).get_indexer(g["unit"]); ci = pd.Index(SET).get_indexer(g["cat"].astype(int))
    n = np.zeros((len(units), len(SET))); np.add.at(n, (ui, ci), 1)
    S = {q: np.zeros((len(units), len(SET))) for q in MEASURES}
    for q in MEASURES:
        np.add.at(S[q], (ui, ci), g[q].to_numpy(dtype=float))
    return n, S


UCS = {ds: unit_cat_sums(ds) for ds in DATASETS}
sem_arr = np.asarray(SEM); doms = sorted(set(SEM))


def domain_means(n, S, w=None):
    w = np.ones(len(n)) if w is None else w
    N = w @ n; out = {}
    for q in MEASURES:
        cm = (w @ S[q]) / np.where(N > 0, N, np.nan)
        out[q] = {**{dd: np.nanmean(cm[sem_arr == dd]) for dd in doms}, "all": np.nanmean(cm)}
    return out, cm


rng = np.random.default_rng(3)
obs = {ds: domain_means(*UCS[ds])[0] for ds in DATASETS}
draws = {ds: [] for ds in DATASETS}
for _ in range(N_BOOT):
    for ds in DATASETS:
        n, S = UCS[ds]; w = rng.multinomial(len(n), np.full(len(n), 1 / len(n))).astype(float)
        draws[ds].append(domain_means(n, S, w)[0])
rows = []
for q in MEASURES:
    for dd in doms + ["all"]:
        v = {ds: np.array([dr[q][dd] for dr in draws[ds]]) for ds in DATASETS}
        lo, hi, p = ci_p(v["bv"] - v["vedb"])
        rows.append({"measure": q, "cdi_semantic": dd, "n_categories": int((sem_arr == dd).sum()) if dd != "all" else len(SET),
                     "bv": obs["bv"][q][dd], "vedb": obs["vedb"][q][dd],
                     "bv_ci_lo": np.percentile(v["bv"], 2.5), "bv_ci_hi": np.percentile(v["bv"], 97.5),
                     "vedb_ci_lo": np.percentile(v["vedb"], 2.5), "vedb_ci_hi": np.percentile(v["vedb"], 97.5),
                     "bv_minus_vedb": obs["bv"][q][dd] - obs["vedb"][q][dd], "diff_ci_lo": lo, "diff_ci_hi": hi, "p_unit_boot": p})
geo_domain = pd.DataFrame(rows)
geo_domain.to_csv(GEO_RESULTS / "geometry_per_domain.csv", index=False)

percat = []
for ds in DATASETS:
    g = GEO[ds][GEO[ds]["cat"].isin(SET)]
    agg = g.groupby("cat").agg(n=("cat", "size"), **{f"mean_{q}": (q, "mean") for q in MEASURES},
                               median_log_area=("log_area", "median"), median_short_px=("short_px", "median"))
    agg.insert(0, "dataset", ds); agg.insert(1, "category", [CATS[i] for i in agg.index])
    agg.insert(2, "cdi_semantic", [CDI.get(CATS[i], "other") for i in agg.index]); percat.append(agg)
geo_cat = pd.concat(percat, ignore_index=True)
geo_cat.to_csv(GEO_RESULTS / "geometry_per_category.csv", index=False)
display(geo_domain[geo_domain.cdi_semantic == "all"].round(3))

## 2 · AUC within geometry levels

Category means from only the crops in one level of a measure. Categories need ≥ `MIN_LEVEL` crops in every level of
that measure in both datasets, so each measure's levels share one category set.

In [ ]:
LEVELS = {"size": {"small": [0, 1], "medium": [2, 3], "large": [4, 5]},
          "vertical position": {"top third": [0], "middle third": [1], "bottom third": [2]},
          "truncation": {"inside frame": [0], "truncated": [1]}}


def cell_mask(measure, level_bins):
    k = np.arange(K_CELLS); s = k // (N_VPOS * 2); v = (k // 2) % N_VPOS; t = k % 2
    axis = {"size": s, "vertical position": v, "truncation": t}[measure]
    return np.isin(axis, level_bins)


lvl_rows = []
for m in MODELS:
    for meas, levels in LEVELS.items():
        masks = {lv: cell_mask(meas, b) for lv, b in levels.items()}
        cnt = {(ds, lv): DN[(ds, m)]["CKn"][:, mk].sum(1) for ds in DATASETS for lv, mk in masks.items()}
        keep = np.flatnonzero(np.logical_and.reduce([cnt[key][SET] >= MIN_LEVEL for key in cnt]))
        sem = [SEM[i] for i in keep]; cats_ix = SET[keep]
        if len(set(sem)) < 2:
            continue
        boot = domain_bootstrap(sem, N_BOOT, 21)
        R = {}
        for ds in DATASETS:
            for lv, mk in masks.items():
                d = DN[(ds, m)]; M = d["CK"][cats_ix][:, mk].sum(1) / cnt[(ds, lv)][cats_ix, None]
                R[(ds, lv)] = rdm(M)
        bd = {key: np.array([auc(R[key], sem, ix) for ix in boot]) for key in R}
        for (ds, lv), Rr in R.items():
            lvl_rows.append({"model": m, "measure": meas, "level": lv, "dataset": ds, "n_categories": len(keep),
                             "auc": auc(Rr, sem), "ci_lo": np.percentile(bd[(ds, lv)], 2.5),
                             "ci_hi": np.percentile(bd[(ds, lv)], 97.5), "n_crops": int(cnt[(ds, lv)][cats_ix].sum())})
        for lv in levels:
            lo, hi, p = ci_p(bd[("bv", lv)] - bd[("vedb", lv)])
            lvl_rows.append({"model": m, "measure": meas, "level": lv, "dataset": "bv_minus_vedb",
                             "n_categories": len(keep), "auc": auc(R[("bv", lv)], sem) - auc(R[("vedb", lv)], sem),
                             "ci_lo": lo, "ci_hi": hi, "p_boot": p})
        first, last = list(levels)[0], list(levels)[-1]
        for ds in DATASETS:
            lo, hi, p = ci_p(bd[(ds, last)] - bd[(ds, first)])
            lvl_rows.append({"model": m, "measure": meas, "level": f"{last} minus {first}", "dataset": ds,
                             "n_categories": len(keep), "auc": auc(R[(ds, last)], sem) - auc(R[(ds, first)], sem),
                             "ci_lo": lo, "ci_hi": hi, "p_boot": p})
auc_levels = pd.DataFrame(lvl_rows)
auc_levels.to_csv(GEO_RESULTS / "auc_by_geometry_level.csv", index=False)
display(auc_levels.pivot_table(index=["model", "measure", "level"], columns="dataset", values="auc", sort=False).round(3))

## 3 · Geometry matching

Per category, common weights over the 36 size × position × truncation cells (the average of BV's and VEDB's cell
shares, over cells where both have ≥ `MIN_CELL` crops). Each dataset's matched mean is the weighted mean of its cell
means. Coverage = share of a dataset's crops in the cells used.

In [ ]:
def match_weights(m):
    nb, nv = DN[("bv", m)]["CKn"], DN[("vedb", m)]["CKn"]
    use = (nb >= MIN_CELL) & (nv >= MIN_CELL)
    pb = nb / np.maximum(nb.sum(1, keepdims=True), 1); pv = nv / np.maximum(nv.sum(1, keepdims=True), 1)
    w = np.where(use, (pb + pv) / 2, 0.0); w = w / np.where(w.sum(1, keepdims=True) > 0, w.sum(1, keepdims=True), 1)
    cov = {ds: (DN[(ds, m)]["CKn"] * use).sum(1) / np.maximum(DN[(ds, m)]["CKn"].sum(1), 1) for ds in DATASETS}
    nused = {ds: (DN[(ds, m)]["CKn"] * use).sum(1) for ds in DATASETS}
    return w, use, cov, nused


def matched_means(CK, CKn, w):
    has = CKn > 0; ww = np.where(has, w, 0.0); ww = ww / np.where(ww.sum(1, keepdims=True) > 0, ww.sum(1, keepdims=True), 1)
    return np.einsum("ck,ckd->cd", ww, CK / np.maximum(CKn, 1)[..., None])


MATCH = {m: match_weights(m) for m in MODELS}
mok = np.logical_and.reduce([MATCH[m][3][ds][SET] >= MIN_MATCHED for m in MODELS for ds in DATASETS])
MSET = SET[mok]; MSEM = [SEM[i] for i in np.flatnonzero(mok)]
MBOOT = domain_bootstrap(MSEM, N_BOOT, 31)
print(f"{len(MSET)} of {len(SET)} categories have ≥ {MIN_MATCHED} matched crops in both datasets")

RD = {}
match_rows, mdiff_rows = [], []
for m in MODELS:
    w = MATCH[m][0][MSET]
    for ds in DATASETS:
        d = DN[(ds, m)]
        RD[(m, ds, "unmatched")] = rdm(d["UC"].sum(0)[MSET] / d["UCn"].sum(0)[MSET, None])
        RD[(m, ds, "matched")] = rdm(matched_means(d["CK"][MSET], d["CKn"][MSET], w))
    bd = {key[1:]: np.array([auc(RD[key], MSEM, ix) for ix in MBOOT]) for key in RD if key[0] == m}
    br = {cond: np.array([rdm_rho(RD[(m, "bv", cond)], RD[(m, "vedb", cond)], ix) for ix in MBOOT])
          for cond in ("unmatched", "matched")}
    for cond in ("unmatched", "matched"):
        rho = rdm_rho(RD[(m, "bv", cond)], RD[(m, "vedb", cond)])
        for ds in DATASETS:
            match_rows.append({"model": m, "condition": cond, "dataset": ds, "n_categories": len(MSET),
                               "auc": auc(RD[(m, ds, cond)], MSEM), "ci_lo": np.percentile(bd[(ds, cond)], 2.5),
                               "ci_hi": np.percentile(bd[(ds, cond)], 97.5),
                               "median_coverage": float(np.median(MATCH[m][2][ds][MSET])),
                               "rho_bv_vedb": rho, "rho_ci_lo": np.percentile(br[cond], 2.5),
                               "rho_ci_hi": np.percentile(br[cond], 97.5)})
        lo, hi, p = ci_p(bd[("bv", cond)] - bd[("vedb", cond)])
        mdiff_rows.append({"model": m, "contrast": f"bv_minus_vedb_{cond}", "diff": auc(RD[(m, "bv", cond)], MSEM)
                           - auc(RD[(m, "vedb", cond)], MSEM), "ci_lo": lo, "ci_hi": hi, "p_boot": p})
    for ds in DATASETS:
        lo, hi, p = ci_p(bd[(ds, "matched")] - bd[(ds, "unmatched")])
        mdiff_rows.append({"model": m, "contrast": f"{ds}_matched_minus_unmatched", "diff": auc(RD[(m, ds, "matched")], MSEM)
                           - auc(RD[(m, ds, "unmatched")], MSEM), "ci_lo": lo, "ci_hi": hi, "p_boot": p})
    lo, hi, p = ci_p(br["matched"] - br["unmatched"])
    mdiff_rows.append({"model": m, "contrast": "rho_matched_minus_unmatched",
                       "diff": rdm_rho(RD[(m, "bv", "matched")], RD[(m, "vedb", "matched")])
                       - rdm_rho(RD[(m, "bv", "unmatched")], RD[(m, "vedb", "unmatched")]), "ci_lo": lo, "ci_hi": hi, "p_boot": p})
match_summary = pd.DataFrame(match_rows); match_diffs = pd.DataFrame(mdiff_rows)
match_summary.to_csv(GEO_RESULTS / "matching_summary.csv", index=False)
match_diffs.to_csv(GEO_RESULTS / "matching_differences.csv", index=False)
display(match_summary.round(3)); display(match_diffs.round(4))

## 4 · Noise ceilings

Split-half reliability of each RDM: units split in half (`K_SPLITS` splits, notebook 11 seeds), each half's category
means z-scored separately, Spearman ρ between the two half RDMs, Spearman–Brown corrected to full data.
Ceiling for the BV–VEDB correlation = √(rel_BV · rel_VEDB). Matched halves use the full-data matching weights.

In [ ]:
def half_sums(ds, m, a_units):
    u, c, k, n, S = DN[(ds, m)]["rows"]; inA = a_units[u]
    out = {}
    for half, sel in (("A", inA), ("B", ~inA)):
        key = c[sel] * K_CELLS + k[sel]; C = len(CATS)
        CK = group_sum(key, S[sel], C * K_CELLS).reshape(C, K_CELLS, -1).astype(np.float64)
        CKn = np.bincount(key, weights=n[sel], minlength=C * K_CELLS).reshape(C, K_CELLS); out[half] = (CK, CKn)
    return out


ceil_rows = []
for m in MODELS:
    w = MATCH[m][0][MSET]; rel = {}
    for ds in DATASETS:
        rr = {"unmatched": [], "matched": []}
        for k in range(K_SPLITS):
            hs = half_sums(ds, m, split_halves(DN[(ds, m)]["units"], k))
            R = {}
            for half, (CK, CKn) in hs.items():
                CK, CKn = CK[MSET], CKn[MSET]
                R[("unmatched", half)] = rdm(CK.sum(1) / np.maximum(CKn.sum(1), 1)[:, None])
                R[("matched", half)] = rdm(matched_means(CK, CKn, w))
            for cond in rr:
                rr[cond].append(rdm_rho(R[(cond, "A")], R[(cond, "B")]))
        for cond, v in rr.items():
            r_half = float(np.mean(v)); rel[(ds, cond)] = 2 * r_half / (1 + r_half)
            ceil_rows.append({"model": m, "condition": cond, "dataset": ds, "split_half_rho": r_half,
                              "split_half_rho_sd": float(np.std(v)), "reliability_sb": rel[(ds, cond)]})
    for cond in ("unmatched", "matched"):
        rho = rdm_rho(RD[(m, "bv", cond)], RD[(m, "vedb", cond)]); ceil = np.sqrt(rel[("bv", cond)] * rel[("vedb", cond)])
        ceil_rows.append({"model": m, "condition": cond, "dataset": "bv_vs_vedb", "rho_bv_vedb": rho,
                          "noise_ceiling": ceil, "rho_over_ceiling": rho / ceil})
ceilings = pd.DataFrame(ceil_rows)
ceilings.to_csv(GEO_RESULTS / "noise_ceilings.csv", index=False)
display(ceilings.round(3))

## 5 · Equivalence of BV and VEDB AUC (unit × category bootstrap)

Each draw resamples units (BV subjects, VEDB sessions) with replacement, recomputes category means, then resamples
categories within domain. Categories with no crops in a draw are dropped from that draw. Equivalence at α = .05 =
the 90% interval inside ±`EQ_BOUND`. Uses the full category set (unmatched means).

In [ ]:
UCS_SET = {key: (d["UC"][:, SET].reshape(len(d["units"]), -1), d["UCn"][:, SET]) for key, d in DN.items()}


def unit_boot_rdm(ds, m, rng):
    X, Nn = UCS_SET[(ds, m)]; U = len(Nn); w = rng.multinomial(U, np.full(U, 1 / U)).astype(np.float32)
    N = w @ Nn; S = (w @ X).reshape(len(SET), -1)
    return S / np.maximum(N, 1)[:, None], N > 0


eq_rows = []
for m in MODELS:
    rng = np.random.default_rng(55); diffs, dropped = [], 0
    R0 = {ds: rdm(DN[(ds, m)]["UC"].sum(0)[SET] / DN[(ds, m)]["UCn"].sum(0)[SET, None]) for ds in DATASETS}
    obs_d = auc(R0["bv"], SEM) - auc(R0["vedb"], SEM)
    cat_only = np.array([auc(R0["bv"], SEM, ix) - auc(R0["vedb"], SEM, ix) for ix in BOOT])
    for ix in BOOT:
        (Mb, okb), (Mv, okv) = unit_boot_rdm("bv", m, rng), unit_boot_rdm("vedb", m, rng)
        good = okb & okv; dropped += int((~good).any())
        keep = np.flatnonzero(good); remap = -np.ones(len(SET), int); remap[keep] = np.arange(len(keep))
        jx = remap[ix]; jx = jx[jx >= 0]; sem_k = [SEM[i] for i in keep]
        diffs.append(auc(rdm(Mb[keep]), sem_k, jx) - auc(rdm(Mv[keep]), sem_k, jx))
    diffs = np.array(diffs)
    for name, v in (("category bootstrap", cat_only), ("unit × category bootstrap", diffs)):
        lo95, hi95, p = ci_p(v, 95); lo90, hi90, _ = ci_p(v, 90)
        eq_rows.append({"model": m, "resampling": name, "n_categories": len(SET), "bv_minus_vedb_auc": obs_d,
                        "ci95_lo": lo95, "ci95_hi": hi95, "ci90_lo": lo90, "ci90_hi": hi90, "p_two_sided": p,
                        "eq_bound": EQ_BOUND, "equivalent": bool(lo90 > -EQ_BOUND and hi90 < EQ_BOUND),
                        "draws_with_dropped_categories": dropped if name != "category bootstrap" else 0})
equivalence = pd.DataFrame(eq_rows)
equivalence.to_csv(GEO_RESULTS / "equivalence_bv_vedb_auc.csv", index=False)
display(equivalence.round(4))

## Figures

In [ ]:
def style(ax):
    ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)


DOMAIN_COLORS = {"animals": "#4DB8A8", "body_parts": "#E87A5F", "clothing": "#9B7EC8", "food_drink": "#E8A54C", "furniture_rooms": "#6BAB7A", "household": "#D97B9E", "outside": "#5B9BD5", "people": "#E8C44C", "toys": "#B07CC8", "vehicles": "#6BA3D5", "other": "#8B9A9E"}


def fig_geometry():
    qs = ["log_area", "cy", "trunc", "n_frame_boxes"]; rows = doms + ["all"]
    fig, axes = plt.subplots(1, len(qs), figsize=(13, 0.32 * len(rows) + 1.6), sharey=True, layout="constrained")
    for ax, q in zip(axes, qs):
        t = geo_domain[geo_domain.measure == q].set_index("cdi_semantic")
        for y, dd in enumerate(rows):
            for ds, off in (("bv", -0.15), ("vedb", 0.15)):
                ax.plot([t.loc[dd, f"{ds}_ci_lo"], t.loc[dd, f"{ds}_ci_hi"]], [y + off] * 2, color=COLOR[ds], lw=1.4)
                ax.scatter([t.loc[dd, ds]], [y + off], s=22, color=COLOR[ds], zorder=3)
        ax.set_yticks(range(len(rows)), [r.replace("_", " ") for r in rows]); ax.set_ylim(len(rows) - 0.5, -0.5)
        for tk, r in zip(ax.get_yticklabels(), rows):    # CDI domains in the manuscript colours
            tk.set_color(DOMAIN_COLORS.get(r, "#222222"))
        ax.set_title(MEASURES[q], fontsize=10); style(ax)
    handles = [plt.Line2D([], [], marker="o", color=COLOR[ds], label=LABEL[ds]) for ds in DATASETS]
    fig.legend(handles=handles, loc="outside lower center", ncol=2, frameon=False)
    fig.suptitle(f"Crop geometry by CDI domain ({len(SET)} categories; 95% unit-bootstrap CI)", fontsize=11)
    out = GEO_FIGURES / "geometry_by_domain.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


def fig_levels():
    fig, axes = plt.subplots(len(MODELS), len(LEVELS), figsize=(12, 6.2), sharey=True, layout="constrained")
    for i, m in enumerate(MODELS):
        for j, meas in enumerate(LEVELS):
            ax = axes[i, j]; t = auc_levels[(auc_levels.model == m) & (auc_levels.measure == meas)]
            lv = list(LEVELS[meas])
            for ds, off in (("bv", -0.06), ("vedb", 0.06)):
                s = t[t.dataset == ds].set_index("level").reindex(lv); x = np.arange(len(lv)) + off
                ax.errorbar(x, s.auc, yerr=[s.auc - s.ci_lo, s.ci_hi - s.auc], color=COLOR[ds], marker="o", capsize=3,
                            label=LABEL[ds])
            ax.axhline(0.5, color="0.6", lw=0.8, ls=":")
            ax.set_xticks(range(len(lv)), lv, fontsize=9)
            n = t.n_categories.iloc[0] if len(t) else 0
            ax.set_title(f"{MLABEL[m]} · {meas} ({n} categories)", fontsize=10); ax.grid(axis="y", color="0.9")
            for sp in ("top", "right"):
                ax.spines[sp].set_visible(False)
        axes[i, 0].set_ylabel("AUC")
    axes[0, 0].legend(frameon=False, fontsize=9)
    fig.suptitle("Domain clustering within geometry levels (95% category-bootstrap CI)", fontsize=11)
    out = GEO_FIGURES / "auc_by_geometry_level.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


def fig_matching():
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), layout="constrained")
    ax = axes[0]; labels = []
    for y, (m, ds) in enumerate([(m, ds) for m in MODELS for ds in DATASETS]):
        s = match_summary[(match_summary.model == m) & (match_summary.dataset == ds)].set_index("condition")
        ax.plot([s.loc["unmatched", "auc"], s.loc["matched", "auc"]], [y, y], color="0.6", lw=2)
        ax.scatter([s.loc["unmatched", "auc"]], [y], s=60, facecolor="white", edgecolor=COLOR[ds], lw=1.5, zorder=3)
        ax.scatter([s.loc["matched", "auc"]], [y], s=60, color=COLOR[ds], zorder=4)
        labels.append(f"{MLABEL[m]} · {LABEL[ds]}")
    ax.set_yticks(range(len(labels)), labels); ax.set_ylim(len(labels) - 0.5, -0.5)
    ax.set_xlim(0.70, 0.80); ax.set_xlabel("AUC (hollow = all crops, filled = geometry-matched)"); style(ax)
    ax = axes[1]
    for y, m in enumerate(MODELS):
        for cond, off, fc in (("unmatched", -0.12, "white"), ("matched", 0.12, "0.3")):
            c = ceilings[(ceilings.model == m) & (ceilings.condition == cond) & (ceilings.dataset == "bv_vs_vedb")].iloc[0]
            ax.scatter([c.rho_bv_vedb], [y + off], s=60, facecolor=fc, edgecolor="0.2", zorder=3)
            ax.plot([c.noise_ceiling] * 2, [y + off - 0.08, y + off + 0.08], color="#b35a0e", lw=2.5)
    ax.set_yticks(range(len(MODELS)), [MLABEL[m] for m in MODELS]); ax.set_ylim(len(MODELS) - 0.5, -0.5)
    ax.set_xlim(0.70, 1.0); ax.set_xlabel("BV–VEDB RDM ρ (dots) vs noise ceiling (orange)"); style(ax)
    fig.suptitle(f"Geometry matching ({len(MSET)} categories)", fontsize=11)
    out = GEO_FIGURES / "geometry_matching.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig); return out


show_images([fig_geometry(), fig_levels(), fig_matching()], width=900)

### Review checklist

- Do crop counts match notebook 11's cache exactly, and do category means agree to ~1e-6?
- Are the size bins balanced enough in both datasets (`geometry_size_bins.csv`), or does one dataset fill only a few?
- How much of each category survives matching (`median_coverage`)? Low coverage means the matched result rests on
  a narrow slice of views.
- Is the BV–VEDB correlation close to its noise ceiling before matching? If it is, matching has little room to help.
- Equivalence: does the unit × category interval stay inside ±`EQ_BOUND`? BV has only 22 subjects, so this
  interval is the honest one.
- Caveats: geometry is partly the detector's (YOLOE boxes on large, cut-off objects), and the CLIP filter may pass
  some views more often in one dataset; the two cameras' fields of view differ, so positions are not identical
  retinal positions.